# 🔍 Autoencoder (AE) — Practice Notebook

**This notebook contains guided exercises — implement the # TODO blocks.**

**Difficulty**: ⭐⭐ Intermediate  
**Time**: ~60 minutes

---


## 🎯 Section 1: Overview

An **Autoencoder** is a type of artificial neural network used to learn efficient data codings in an unsupervised manner. The aim of an autoencoder is to learn a representation (encoding) for a set of data, typically for dimensionality reduction, by training the network to ignore signal noise.


## 📐 Section 2: Math & Intuition

### Subspace Projection
An Autoencoder consists of two functions:
- Encoder: $z = f(x) = \sigma(W_e x + b_e)$
- Decoder: $\hat{x} = g(z) = \sigma(W_d z + b_d)$

The network is optimized using Mean Squared Error (MSE) reconstruction loss:
$$L = \frac{1}{m} \sum_{i=1}^m \|x^{(i)} - \hat{x}^{(i)}\|_2^2$$

### Relationship to PCA
If the encoder and decoder activations are linear, the bottleneck space spans the same subspace as **Principal Component Analysis (PCA)**.


## 🔧 Section 3: Implementation from Scratch


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)
print('Autoencoder Setup complete! ✅')


### 3.1 Linear Autoencoder vs PCA


In [ ]:
class LinearAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        # TODO: Define linear encoder and linear decoder layers
        
    def forward(self, x):
        # TODO: Implement forward pass
        


In [ ]:
# Generate simple correlated 2D data
np.random.seed(42)
x1 = np.random.randn(300)
x2 = x1 * 2.0 + np.random.randn(300) * 0.2
data = np.column_stack((x1, x2))
data_t = torch.FloatTensor(data)

model = LinearAutoencoder(input_dim=2, latent_dim=1)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.05)

if 'TODO' not in LinearAutoencoder.__init__.__code__.co_consts:
    # Train
    for epoch in range(150):
        optimizer.zero_grad()
        reconstructed = model(data_t)
        loss = criterion(reconstructed, data_t)
        loss.backward()
        optimizer.step()
        
    # Get reconstructed coordinates
    with torch.no_grad():
        recon = model(data_t).numpy()
        
    # Plot original vs. reconstructed
    plt.figure(figsize=(8, 5))
    plt.scatter(data[:, 0], data[:, 1], label='Original', alpha=0.5)
    plt.scatter(recon[:, 0], recon[:, 1], label='Reconstructed', color='red', alpha=0.5)
    plt.title('Linear Autoencoder Projection')
    plt.legend()
    plt.show()
    print('Subspace projection completed! ✅')


## 📦 Section 4: Library Implementation


We will build a Denoising Autoencoder using non-linear layers. Denoising autoencoders learn to reconstruct the clean input from a corrupted version.


In [ ]:
class DenoisingAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()
        # TODO: Define encoder (Linear -> ReLU -> Linear -> ReLU) and decoder
        
    def forward(self, x):
        # TODO: Implement forward pass
        


In [ ]:
dae = DenoisingAutoencoder(input_dim=10, hidden_dim=6, latent_dim=2)
dummy_x = torch.randn(4, 10)
out = dae(dummy_x)
print('DAE output shape (should be [4, 10]):', list(out.shape))
assert list(out.shape) == [4, 10]
print('Denoising Autoencoder structure verified! ✅')


## 🧪 Section 5: Experiments


Let's train our Denoising Autoencoder to clean up synthetic patterns with Gaussian noise.


In [ ]:
# Generate synthetic target pattern
np.random.seed(42)
clean_patterns = np.sin(np.linspace(0, 2*np.pi, 100))[None, :] * np.random.uniform(0.5, 2.0, (200, 1))
clean_t = torch.FloatTensor(clean_patterns)

# Add noise to inputs
noisy_patterns = clean_patterns + np.random.randn(*clean_patterns.shape) * 0.3
noisy_t = torch.FloatTensor(noisy_patterns)

dae = DenoisingAutoencoder(input_dim=100, hidden_dim=32, latent_dim=8)
opt = optim.Adam(dae.parameters(), lr=0.01)
crit = nn.MSELoss()

if 'TODO' not in DenoisingAutoencoder.__init__.__code__.co_consts:
    for epoch in range(250):
        opt.zero_grad()
        recon = dae(noisy_t)
        l = crit(recon, clean_t)  # Learn to output clean patterns
        l.backward()
        opt.step()
        
    with torch.no_grad():
        cleaned = dae(noisy_t).numpy()
        
    # Plot one sample comparison
    plt.figure(figsize=(10, 5))
    plt.plot(clean_patterns[0], label='Original Clean', color='black', linewidth=2)
    plt.plot(noisy_patterns[0], label='Noisy Input', alpha=0.5, linestyle='--')
    plt.plot(cleaned[0], label='Denoised Output', color='green', linewidth=2)
    plt.title('Denoising Autoencoder Verification')
    plt.legend()
    plt.show()


## ❓ Section 6: Interview Questions


### Q1: What is the relationship between a linear autoencoder and Principal Component Analysis (PCA)?
**Answer**:
Under linear activations and an MSE loss function, the hidden representation of a bottleneck autoencoder learns to project input vectors into a subspace spanned by the first $K$ principal components. However, unlike standard PCA, the autoencoder weights are not required to be orthogonal, so it learns a scaled and rotated representation of the same subspace.

### Q2: Why is a non-linear activation necessary in autoencoders?
**Answer**:
If all activation functions are linear, the network can only represent projections into linear subspaces, meaning the autoencoder is functionally identical to PCA. Non-linear activation functions (such as ReLU or Sigmoid) allow the network to learn complex non-linear coordinate projections, capturing highly non-linear manifolds in data.

### Q3: What is a Denoising Autoencoder (DAE) and how does it prevent the model from learning the identity function?
**Answer**:
A Denoising Autoencoder is trained to reconstruct the original clean input $x$ from a deliberately corrupted input $\tilde{x} = x + \epsilon$. Because the input is distorted, the model cannot simply learn the identity function (copying input to output). Instead, it must capture the underlying distribution structure of the clean data in its latent space to clean up the inputs.

### Q4: Explain the reconstruction threshold trick for anomaly detection.
**Answer**:
To detect anomalies, an Autoencoder is trained strictly on normal data instances. Because it only learns representations for normal instances, it will reconstruct normal validation items with low MSE. When presented with an anomalous instance, the reconstruction error will be significantly higher because it falls outside the learned manifold. We choose an MSE threshold (e.g. 95th percentile of normal training loss) and flag any item exceeding it as an anomaly.


## 🏆 Section 7: Challenge — Anomaly Detection


**Challenge**: Implement the anomaly detection decision logic: compute reconstruction MSE per instance and flag inputs as anomalous if they exceed a specific threshold.


In [ ]:
def detect_anomalies(original, reconstructed, threshold):
    """
    original: numpy array of shape (N, D)
    reconstructed: numpy array of shape (N, D)
    threshold: scalar representation of maximum allowed MSE
    Return boolean array of shape (N,) where True = Anomaly
    """
    # TODO: Implement anomaly classification logic
    
# Test
orig = np.array([[1.0, 1.0], [1.0, 1.0], [1.0, 10.0]])  # Last one is anomalous
recon = np.array([[1.0, 1.1], [1.0, 0.9], [1.0, 1.0]])
anom = detect_anomalies(orig, recon, 0.5)
print('Anomaly Flags:', anom)
if 'TODO' not in detect_anomalies.__code__.co_consts:
    assert list(anom) == [False, False, True]
    print('Anomaly Detection logic works! ✅')
